# 04. KRX 수정주가 · 시장지수 수집 (pykrx)

**이 노트북이 하는 일**: 이벤트 스터디용 일별 수정 종가와 시장지수를 모읍니다. 매핑표의 시장구분대로 **유가 종목은 코스피, 코스닥 종목은 코스닥 지수**와 짝짓습니다. 휴장일·거래정지·상장 전을 나눠 표시하고 거래정지일 가격은 비워 둡니다(그 기간이 들어간 이벤트는 계산에서 빠짐).

- 준비물: KRX 정보데이터시스템 아이디·비밀번호(pykrx가 로그인을 요구함), 01 담당자에게 받은 `company_map.csv`
- 결과: 내 드라이브 `ONE_PICK_결과/`에 zip 하나(`prices.csv`(행: 거래일, 열: corp_code), `market_KOSPI.csv`, `market_KOSDAQ.csv`, `종목별_시장.csv`, `report.csv`, `일별상태.csv`)
- 순서: 01 매핑표 → **04 KRX**

**실행 방법**: 아래 **0. 설정**만 확인하고 메뉴 **런타임 → 모두 실행**. 중간에 드라이브 연결 허용 창과 키 입력 칸이 뜨면 답하면 됩니다.
KIND 목록이나 `company_map.csv`는 내 드라이브 `ONE_PICK_결과/`에 넣어 두거나 코랩 왼쪽 📁에 올리면 자동으로 찾습니다(없으면 업로드 창이 뜸).
연결·키에 문제가 있으면 **2번에서 멈추고** 고칠 방법을 알려 줍니다.

**결과 넘기기**: zip은 **내 드라이브 `ONE_PICK_결과/`**에 생깁니다(폴더가 없으면 만듦). 팀에 넘길 때는 그 zip을 공유 폴더나 메신저에 직접 올려 주세요.


| 상태 | 뜻 | 할 일 |
|---|---|---|
| 정상 | API에서 값을 찾음 | 그대로 사용 |
| 원문에서 찾음 | API에는 없고 공시 원문 표에서 찾음 | 원문 링크로 한 번 더 확인 |
| 보관본 사용 | API 대신 미리 받아 둔 사본 | 받은 날짜 확인 |
| 데이터 없음 | 원천이 '값 없음'이라고 응답(`013`, `'-'`) | 값을 채우지 말고 '확인 필요'로 둠 |
| 조회 실패 | 키·한도·점검·네트워크 문제 | 나중에 이 노트북을 다시 실행 |
| 확인 불가 | 모든 경로를 찾았지만 없거나 후보가 여러 개 | 원문 확인 목록으로 넘김 |

값이 비면 **0이나 추정값으로 채우지 않습니다.** 메모와 시도 경로를 그대로 남깁니다.

> 한국거래소(KRX) 시세를 pykrx로 받습니다. 실서비스로 바꿀 때 시세 이용계약은 PRD 6.1을 따릅니다.
> pykrx 1.2 이후는 KRX 로그인이 없으면 빈 값을 줍니다. KRX 정보데이터시스템(data.krx.co.kr) 계정을 비밀키 `KRX_ID`·`KRX_PW`로 넣어 주세요.

## 0. 설정 (여기만 확인)

In [ ]:
# @title 0. 설정
EXPORT_FOLDER = "ONE_PICK_결과"  # @param {type:"string"}
MAP_FILE = "company_map.csv"  # @param {type:"string"}
NB_NAME = "04_KRX시세"
START = "2018-01-01"  # @param {type:"string"}
END = "2026-10-20"  # @param {type:"string"}

## 1. 코드 준비 (손댈 것 없음)

In [ ]:
# @title 1-0. 패키지 설치
%pip install -q pykrx

In [ ]:
# @title 1-1. 수집 코드 파일 만들기
from pathlib import Path
Path("core/sources").mkdir(parents=True, exist_ok=True)
Path("core/__init__.py").write_text("")

In [ ]:
%%writefile core/sources/__init__.py
"""외부 자료 수집과 결측 보완.

화면에는 보이는데 API가 빈 값을 주는 경우를 줄이고, 끝까지 못 찾은 값은 채우지 않고 사유와 함께 남긴다.

- base: 결측 상태, 숫자 변환, 재시도·원본 캐시, 출처 기록, 이어받기
- opendart: OpenDART (재무·배당·공시 검색·원문·주식 총수·최대주주)
- krx: KRX 수정주가·시장지수 (pykrx)
- law: 국가법령정보센터 법령 원문
- kind: KIND 고배당기업 목록 엑셀
"""

In [ ]:
%%writefile core/sources/base.py
"""수집 공통 부품: 결측 상태, 숫자 변환, 재시도·원본 캐시, 출처 기록, 이어받기.

원칙: 값을 못 찾으면 0이나 추정값으로 채우지 않는다. 왜 없는지(상태)와 어디까지 찾아봤는지(시도 경로)를 남긴다.
"""
import hashlib
import json
import re
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable, Iterable

import pandas as pd

OK = "정상"
NO_DATA = "데이터 없음"  # 원천이 '값 없음'이라고 응답함
API_ERROR = "조회 실패"  # 키·한도·점검·네트워크 문제. 데이터가 없다는 뜻이 아님
FROM_ORIGINAL = "원문에서 찾음"  # API에는 없고 공시 원문 표에서 찾음. 사람이 원문을 한 번 더 확인
STORED = "보관본 사용"  # API 대신 미리 저장해 둔 사본을 씀
UNVERIFIED = "확인 불가"  # 모든 경로를 시도했지만 값이 없거나 하나로 정해지지 않음

STATUSES = [OK, NO_DATA, API_ERROR, FROM_ORIGINAL, STORED, UNVERIFIED]

# 캐시 파일 이름·출처 기록에 남기지 않는 비밀 파라미터
SECRET_PARAMS = ("crtfc_key", "OC", "serviceKey")

DEFAULT_CACHE_DIR = Path("data/raw")  # .gitignore에 들어 있어 커밋되지 않음


@dataclass(frozen=True)
class Source:
    """값 하나의 출처. params에는 비밀값을 넣지 않는다."""

    api: str
    params: dict = field(default_factory=dict)
    rcept_no: str | None = None
    url: str | None = None
    note: str = ""


@dataclass
class Fetched:
    """수집 결과 하나. value가 None이면 status가 이유를 말한다.

    unit: value의 단위(원). 백만 원 단위 값이면 1_000_000. 원 단위로 바꾸려면 value × unit.
    attempts: 시도한 경로와 결과를 순서대로 적은 기록.
    """

    value: Any
    status: str
    source: Source | None = None
    attempts: list[str] = field(default_factory=list)
    unit: int = 1

    @property
    def found(self) -> bool:
        return self.value is not None and self.status in (OK, FROM_ORIGINAL, STORED)

    @property
    def won(self) -> int | float | None:
        """원 단위 값. 숫자가 아니면 None."""
        if isinstance(self.value, (int, float)) and not isinstance(self.value, bool):
            return self.value * self.unit
        return None


def public_params(params: dict) -> dict:
    return {k: v for k, v in params.items() if k not in SECRET_PARAMS}


# --- 비밀키 입력 ---

_SECRETS: dict[str, str] = {}  # 이번 실행 동안만 메모리에 둔다. 파일에 쓰지 않는다.


def _colab_secret(name: str) -> str | None:
    try:
        from google.colab import userdata  # type: ignore[import-not-found]
    except ImportError:
        return None
    try:
        return userdata.get(name)
    except Exception:  # noqa: BLE001 - 보안 비밀이 없거나 접근 허용 전
        return None


def secret(name: str, prompt: str | None = None, ask: Callable[[str], str] | None = None) -> str:
    """API 키를 화면에 보이지 않게 받는다.

    순서: 이번 실행에서 이미 받은 값 → 환경변수 name → Colab 보안 비밀 name → 숨김 입력(getpass).
    받은 키는 메모리에만 두며 출력·캐시 파일·출처 기록에 남기지 않는다.
    """
    import os

    if name in _SECRETS:
        return _SECRETS[name]
    value = os.environ.get(name) or _colab_secret(name)
    if not value:
        if ask is None:
            from getpass import getpass as ask
        value = ask(prompt or f"{name} 입력(화면에 보이지 않음): ")
    value = (value or "").strip()
    if not value:
        raise ValueError(f"{name}가 비어 있습니다.")
    _SECRETS[name] = value
    return value


def forget_secret(name: str) -> None:
    """잘못 입력한 키를 지워 다음 secret() 호출에서 다시 묻게 한다."""
    _SECRETS.pop(name, None)


# --- 숫자·이름 정리 ---

_EMPTY = {"", "-", "−", "–", "N/A", "NA", "nan", "None", "해당없음", "해당사항없음"}


def to_number(raw) -> int | float | None:
    """공시 숫자 문자열 → int 또는 float. 비어 있거나 '-'이면 None (0으로 바꾸지 않는다).

    '1,234' → 1234 · '△1,234'·'(1,234)'·'-1,234' → -1234 · '12.5' → 12.5 · '12.5%' → 12.5
    """
    if raw is None:
        return None
    if isinstance(raw, bool):
        return None
    if isinstance(raw, (int, float)):
        return None if raw != raw else raw  # NaN은 None
    s = re.sub(r"\s", "", str(raw)).replace(",", "").rstrip("%")
    if s in _EMPTY:
        return None
    negative = False
    if s[0] in "△▲":
        negative, s = True, s[1:]
    elif s.startswith("(") and s.endswith(")"):
        negative, s = True, s[1:-1]
    if s[:1] in ("-", "−"):
        negative, s = not negative, s[1:]
    try:
        value = float(s) if "." in s else int(s)
    except ValueError:
        return None
    return -value if negative else value


def squash(text: str) -> str:
    """비교용 이름: 공백과 가운뎃점을 모두 지운다. '현금ㆍ현물 배당결정' → '현금현물배당결정'."""
    return re.sub(r"[\sㆍ·•‧]", "", str(text or ""))


def strip_brackets(text: str) -> str:
    """비교용 이름에서 괄호 덧말과 앞 번호를 지운다. 'Ⅰ. 당기순이익(손실)' → '당기순이익'."""
    s = re.sub(r"\([^)]*\)|\[[^\]]*\]", "", squash(text))
    return re.sub(r"^([0-9]+|[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩ]+|[가-하])\.", "", s)


_UNITS = {"백만원": 1_000_000, "천원": 1_000, "억원": 100_000_000, "원": 1}


def unit_from_label(label: str) -> int:
    """항목 이름 괄호 속 단위 → 원 단위 배수. '현금배당금총액(백만원)' → 1_000_000. 없으면 1."""
    for inner in re.findall(r"\(([^)]*)\)", squash(label)):
        if inner in _UNITS:
            return _UNITS[inner]
    return 1


# --- 재시도와 원본 캐시 ---

Fetch = Callable[[str, dict], bytes]


class FetchError(Exception):
    """재시도해도 응답을 받지 못함."""


def http_fetch(url: str, params: dict, timeout: float = 20) -> bytes:
    import requests

    response = requests.get(url, params=params, timeout=timeout)
    response.raise_for_status()
    return response.content


def scrub(text: str) -> str:
    """오류 메시지 속 URL에 섞인 키를 가린다. 'crtfc_key=abc&' → 'crtfc_key=***&'."""
    return re.sub(rf"({'|'.join(SECRET_PARAMS)})=[^&\s'\"]+", r"\1=***", str(text))


def _is_client_error(error: Exception) -> bool:
    status = getattr(getattr(error, "response", None), "status_code", None)
    return status is not None and 400 <= status < 500


class Client:
    """재시도와 원본 캐시를 붙인 GET.

    - 네트워크 오류·5xx는 backoff × 2^n초 쉬고 다시 시도한다. 4xx는 다시 시도하지 않는다.
    - cache_dir가 있으면 응답 원본을 저장해 두고 다음에는 저장본을 쓴다(호출 한도 절약, 결측 원인 다시 보기).
      keep(응답)이 False면 저장하지 않는다(한도 초과·점검 응답처럼 다시 받아야 하는 경우).
    - 이번 실행 동안은 remember(응답)가 True인 응답을 메모리에 기억해 같은 요청을 다시 보내지 않는다.
      remember를 주지 않으면 keep을 따른다('데이터 없음'처럼 파일로는 남기지 않지만 같은 실행에서는 다시 물을 필요 없는 응답).
    - fetch를 바꿔 끼우면 인터넷 없이 테스트할 수 있다.
    """

    def __init__(
        self,
        fetch: Fetch | None = None,
        cache_dir: Path | str | None = None,
        retries: int = 3,
        backoff: float = 1.0,
        sleep: Callable[[float], None] = time.sleep,
    ):
        self.fetch = fetch or http_fetch
        self.cache_dir = Path(cache_dir) if cache_dir else None
        self.retries = retries
        self.backoff = backoff
        self.sleep = sleep
        self.calls = 0  # 실제로 원천에 보낸 요청 수
        self._memo: dict[str, bytes] = {}

    @staticmethod
    def _key(url: str, params: dict) -> str:
        return json.dumps([url, sorted(public_params(params).items())], ensure_ascii=False, default=str)

    def cache_path(self, url: str, params: dict) -> Path | None:
        if not self.cache_dir:
            return None
        digest = hashlib.sha256(self._key(url, params).encode()).hexdigest()[:20]
        folder = re.sub(r"[^0-9A-Za-z]+", "_", url.split("://")[-1]).strip("_")
        return self.cache_dir / folder / f"{digest}.bin"

    def get(self, url: str, params: dict, *, keep: Callable[[bytes], bool] | None = None,
            remember: Callable[[bytes], bool] | None = None) -> bytes:
        key = self._key(url, params)
        if key in self._memo:
            return self._memo[key]
        path = self.cache_path(url, params)
        if path and path.exists():
            return path.read_bytes()
        last_error: Exception | None = None
        for attempt in range(self.retries):
            try:
                self.calls += 1
                data = self.fetch(url, params)
                break
            except Exception as error:  # noqa: BLE001 - 원천마다 예외 종류가 달라 모두 받는다
                last_error = error
                if _is_client_error(error) or attempt == self.retries - 1:
                    raise FetchError(scrub(f"{url} 요청 실패({attempt + 1}번 시도): {error}")) from None
                self.sleep(self.backoff * 2**attempt)
        else:  # retries가 0인 경우
            raise FetchError(scrub(f"{url} 요청 실패: {last_error}"))
        if path and (keep is None or keep(data)):
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(data)
        remember = remember or keep
        if remember is None or remember(data):
            self._memo[key] = data
        return data

    def forget(self, url: str, params: dict) -> None:
        """저장본을 지워 다음에 원천에서 다시 받게 한다."""
        self._memo.pop(self._key(url, params), None)
        path = self.cache_path(url, params)
        if path and path.exists():
            path.unlink()


# --- 이어받기 (하루 호출 한도) ---


class LimitReached(Exception):
    """하루 호출 한도를 넘음. 다음 날 같은 체크포인트로 이어서 돌린다."""


def run_resumable(keys: Iterable, task: Callable[[Any], Any], checkpoint: Path | str) -> dict:
    """keys마다 task(key)를 부르고, 끝난 key를 checkpoint(JSON)에 적는다.

    LimitReached가 나면 멈추고 남은 key를 돌려준다. 다음에 같은 checkpoint로 부르면 끝난 key는 건너뛴다.
    결과 자체는 저장하지 않는다(응답 원본은 Client 캐시에 남으므로 다시 불러도 호출이 늘지 않는다).
    """
    checkpoint = Path(checkpoint)
    done: set[str] = set(json.loads(checkpoint.read_text())) if checkpoint.exists() else set()
    keys = list(keys)
    results: dict = {}
    stopped = False
    for key in keys:
        if str(key) in done:
            continue
        try:
            results[key] = task(key)
        except LimitReached:
            stopped = True
            break
        done.add(str(key))
        checkpoint.parent.mkdir(parents=True, exist_ok=True)
        checkpoint.write_text(json.dumps(sorted(done), ensure_ascii=False))
    return {
        "results": results,
        "remaining": [k for k in keys if str(k) not in done],
        "stopped_by_limit": stopped,
    }


# --- 수집 현황표 ---


def status_table(items: dict[str, Fetched]) -> pd.DataFrame:
    """항목 이름 → Fetched를 한 표로. 팀 수집 현황표·원문 확인 목록에 쓴다."""
    rows = []
    for name, f in items.items():
        rows.append({
            "항목": name,
            "값": f.value if not isinstance(f.value, pd.DataFrame) else f"표 {len(f.value)}행",
            "단위(원)": f.unit,
            "상태": f.status,
            "출처 API": f.source.api if f.source else None,
            "접수번호": f.source.rcept_no if f.source else None,
            "메모": f.source.note if f.source else "",
            "시도 경로": " → ".join(f.attempts),
        })
    # dtype=object: 정수 값이 결측 행 때문에 실수(4.8e+09)로 바뀌지 않게
    return pd.DataFrame(rows, columns=["항목", "값", "단위(원)", "상태", "출처 API", "접수번호", "메모", "시도 경로"], dtype=object)

In [ ]:
%%writefile core/sources/krx.py
"""KRX 수정주가·시장지수(pykrx) 수집과 결측 구분.

화면(KRX 정보데이터시스템)에는 보이는데 코드에서 비는 흔한 원인과 대응:
- 종목코드 앞자리 0이 빠짐(5930) 또는 'A005930'·'005930.KS' 형식 → normalize_ticker로 6자리
- 휴장일 → 결측이 아님. 거래일 달력을 시장지수 날짜로 만들어 휴장일은 아예 행을 만들지 않음
- 거래정지일 → pykrx는 시가·고가·저가 0, 거래량 0, 종가는 전날 값으로 줌. 정상 가격으로 쓰면 수익률이 0으로 왜곡되므로 비워 둠
- 상장 전·마지막 거래 이후 → '값 없음'과 나눠 표시
- KRX 응답 자체가 비는 경우 → 지수까지 비면 '조회 실패'(데이터 없음과 구분)

비워 둔 날이 관측 기간에 들어간 이벤트는 core.event_study.car가 None을 돌려 사례에서 빠진다(값을 채우지 않음).
자료는 한국거래소(KRX) 시세를 pykrx로 받는다. 실서비스로 바꿀 때 시세 이용계약은 PRD 6.1을 따른다.
"""
import re
from typing import Protocol

import pandas as pd

from core.sources.base import API_ERROR, NO_DATA, OK, Fetched, Source, scrub

MARKET_INDEX = {"KOSPI": "1001", "KOSDAQ": "2001"}
MARKET_NAMES = {"유가": "KOSPI", "유가증권": "KOSPI", "코스피": "KOSPI", "KOSPI": "KOSPI",
                "코스닥": "KOSDAQ", "KOSDAQ": "KOSDAQ"}  # KIND·DART 시장 표기 → pykrx 시장 이름

NORMAL = "정상"
BEFORE_LISTING = "상장 전"
AFTER_LAST_TRADE = "마지막 거래 이후"
HALTED = "거래정지"
MISSING = "값 없음"
DAY_STATUSES = [NORMAL, HALTED, MISSING, BEFORE_LISTING, AFTER_LAST_TRADE]


class PriceProvider(Protocol):
    name: str

    def prices(self, ticker: str, start: str, end: str) -> pd.DataFrame: ...

    def index(self, code: str, start: str, end: str) -> pd.DataFrame: ...

    def tickers(self, date: str, market: str) -> dict[str, str]: ...


class PykrxProvider:
    """pykrx로 수정주가(adjusted=True)와 지수 일별 시세를 받는다. 열: 시가·고가·저가·종가·거래량."""

    name = "pykrx"

    def prices(self, ticker: str, start: str, end: str) -> pd.DataFrame:
        from pykrx import stock

        return stock.get_market_ohlcv(start, end, ticker, adjusted=True)

    def index(self, code: str, start: str, end: str) -> pd.DataFrame:
        from pykrx import stock

        return stock.get_index_ohlcv(start, end, code)

    def tickers(self, date: str, market: str) -> dict[str, str]:
        """그날 상장 종목 {종목코드: 종목 약칭}. market: KOSPI·KOSDAQ."""
        from pykrx import stock

        return {t: stock.get_market_ticker_name(t) for t in stock.get_market_ticker_list(date, market=market)}


def normalize_ticker(code) -> str:
    """5930 · 'A005930' · '005930.KS' → '005930'."""
    digits = re.sub(r"\D", "", str(code).split(".")[0])
    if not digits or len(digits) > 6:
        raise ValueError(f"종목코드로 볼 수 없음: {code!r}")
    return digits.zfill(6)


def _ymd(value) -> str:
    return pd.Timestamp(value).strftime("%Y%m%d")


def fetch_ohlcv(ticker, start, end, provider: PriceProvider | None = None,
                fallback: PriceProvider | None = None) -> Fetched:
    """한 종목의 일별 시세. 비거나 실패하면 fallback 공급자로 한 번 더 받는다."""
    code = normalize_ticker(ticker)
    attempts: list[str] = []
    status = NO_DATA
    for p in [x for x in (provider or PykrxProvider(), fallback) if x is not None]:
        try:
            df = p.prices(code, _ymd(start), _ymd(end))
        except Exception as error:  # noqa: BLE001 - 공급자마다 예외가 다르다
            attempts.append(f"{p.name} {code}: 조회 실패 {scrub(error)}")
            status = API_ERROR
            continue
        if df is None or df.empty:
            attempts.append(f"{p.name} {code}: 기간 안에 거래 기록 없음")
            continue
        df = df.copy()
        df.index = pd.to_datetime(df.index)
        attempts.append(f"{p.name} {code}: {len(df)}일")
        return Fetched(df, OK, Source(p.name, {"ticker": code, "start": _ymd(start), "end": _ymd(end)}), attempts)
    return Fetched(None, status, Source("prices", {"ticker": code}), attempts)


def trading_calendar(start, end, provider: PriceProvider | None = None, market: str = "KOSPI") -> Fetched:
    """시장지수에 값이 있는 날 = 거래일. 지수까지 비면 KRX 응답 문제로 본다."""
    p = provider or PykrxProvider()
    code = MARKET_INDEX[market]
    params = {"index": code, "start": _ymd(start), "end": _ymd(end)}
    try:
        df = p.index(code, _ymd(start), _ymd(end))
    except Exception as error:  # noqa: BLE001
        return Fetched(None, API_ERROR, Source(p.name, params), [f"{market} 지수: 조회 실패 {scrub(error)}"])
    if df is None or df.empty:
        return Fetched(None, API_ERROR, Source(p.name, params, note="지수가 비면 KRX 응답 문제(접속 차단·점검)일 가능성이 큼"),
                       [f"{market} 지수: 빈 응답"])
    df = df.copy()
    df.index = pd.to_datetime(df.index)
    return Fetched(df, OK, Source(p.name, params), [f"{market} 지수: {len(df)}거래일"])


def classify_days(ohlcv: pd.DataFrame | None, calendar: pd.DatetimeIndex) -> pd.DataFrame:
    """거래일마다 종가와 상태. 정상이 아닌 날의 '사용 종가'는 비워 둔다."""
    out = pd.DataFrame(index=calendar)
    if ohlcv is None or ohlcv.empty:
        out["종가"] = float("nan")
        out["상태"] = MISSING
        out["사용 종가"] = float("nan")
        return out
    df = ohlcv.reindex(calendar)
    present = df["종가"].notna()  # 거래정지일도 행은 있다. 행이 처음·마지막으로 있는 날로 상장 기간을 본다
    first, last = (present[present].index.min(), present[present].index.max()) if present.any() else (None, None)
    status = []
    for day, row in df.iterrows():
        if first is None or day < first:
            status.append(BEFORE_LISTING if first is not None else MISSING)
        elif day > last:
            status.append(AFTER_LAST_TRADE)
        elif pd.isna(row["종가"]):
            status.append(MISSING)
        elif row["거래량"] == 0 and row.get("시가", 0) == 0:
            status.append(HALTED)
        elif row["종가"] <= 0:
            status.append(MISSING)
        else:
            status.append(NORMAL)
    out["종가"] = df["종가"]
    out["상태"] = status
    out["사용 종가"] = df["종가"].where(out["상태"] == NORMAL)
    return out


def price_panel(tickers, start, end, provider: PriceProvider | None = None, market: str = "KOSPI",
                fallback: PriceProvider | None = None, key_map: dict | None = None) -> dict:
    """이벤트 스터디용 표. 반환:

    - prices: 거래일 × 종목(key_map이 있으면 corp_code) 수정 종가. 정상이 아닌 날은 비어 있음
    - market: 같은 거래일의 시장지수 종가
    - report: 종목별 상태 일수(수집 현황표)
    - fetched: 종목별 Fetched(시도 경로 확인용)
    """
    cal = trading_calendar(start, end, provider, market)
    if not cal.found:
        raise RuntimeError(f"거래일 달력을 만들지 못함: {' → '.join(cal.attempts)}")
    calendar = cal.value.index
    columns, rows, fetched = {}, [], {}
    for t in tickers:
        code = normalize_ticker(t)
        f = fetch_ohlcv(code, start, end, provider, fallback)
        fetched[code] = f
        days = classify_days(f.value, calendar)
        key = (key_map or {}).get(code, code)
        columns[key] = days["사용 종가"]
        counts = days["상태"].value_counts()
        rows.append({"종목코드": code, "열 이름": key, "수집 상태": f.status,
                     **{s: int(counts.get(s, 0)) for s in DAY_STATUSES}, "시도 경로": " → ".join(f.attempts)})
    return {
        "prices": pd.DataFrame(columns, index=calendar),
        "market": cal.value["종가"],
        "report": pd.DataFrame(rows),
        "fetched": fetched,
    }


def listing(date, markets=("KOSPI", "KOSDAQ"), provider: PriceProvider | None = None) -> Fetched:
    """그날 상장 종목표(종목코드·종목명·시장). 시장 하나라도 비면 KRX 응답 문제로 본다."""
    p = provider or PykrxProvider()
    frames, attempts = [], []
    for m in markets:
        try:
            names = p.tickers(_ymd(date), m)
        except Exception as error:  # noqa: BLE001
            return Fetched(None, API_ERROR, Source(p.name, {"date": _ymd(date)}), attempts + [f"{m}: 조회 실패 {scrub(error)}"])
        if not names:
            return Fetched(None, API_ERROR, Source(p.name, {"date": _ymd(date)}, note="휴장일이면 직전 거래일로 다시"),
                           attempts + [f"{m}: 빈 응답"])
        attempts.append(f"{m}: {len(names)}종목")
        frames.append(pd.DataFrame({"종목코드": list(names), "종목명": list(names.values()), "시장": m}))
    return Fetched(pd.concat(frames, ignore_index=True), OK, Source(p.name, {"date": _ymd(date)}), attempts)


def market_panels(tickers_by_market: dict[str, list], start, end, provider: PriceProvider | None = None,
                  fallback: PriceProvider | None = None, key_map: dict | None = None) -> dict:
    """시장별로 price_panel을 돌려 합친다. 코스닥 종목은 코스닥 지수, 코스피 종목은 코스피 지수와 짝짓는다.

    반환: prices(거래일 × 종목), markets({시장: 지수 종가}), market_of({열 이름: 시장}), report, fetched
    """
    prices, markets, market_of, reports, fetched = [], {}, {}, [], {}
    for market, codes in tickers_by_market.items():
        if not codes:
            continue
        panel = price_panel(codes, start, end, provider, market, fallback, key_map)
        prices.append(panel["prices"])
        markets[market] = panel["market"]
        market_of.update({col: market for col in panel["prices"].columns})
        reports.append(panel["report"].assign(시장=market))
        fetched.update(panel["fetched"])
    return {
        "prices": pd.concat(prices, axis=1).sort_index() if prices else pd.DataFrame(),
        "markets": markets,
        "market_of": market_of,
        "report": pd.concat(reports, ignore_index=True) if reports else pd.DataFrame(),
        "fetched": fetched,
    }

In [ ]:
%%writefile core/sources/samples.py
"""인터넷·키 없이 결측 보완 흐름을 보여 주는 가짜 응답. 테스트와 팀 공유 노트북이 함께 쓴다.

회사 '00000001'(가상 회사 가나다)은 실제 응답에서 자주 보던 결측 모양을 일부러 담았다.
- 연결재무제표 없음(013) → 별도만 있음, 계정 이름이 '당기순이익(손실)', account_id가 '-표준계정코드 미사용-'
- 2024 사업보고서 배당 표: 현금배당금총액 행이 '-' , 주당 현금배당금은 보통주·우선주 두 행
- 2025 사업보고서 배당 표의 전기(frmtrm) 칸에 2024 현금배당금총액이 있음
- 배당 결정 공시 원문: DART XML 표(TD·TE)로만 숫자가 있음
- 기업가치 제고 계획 공시: 원문 파일 없음(014)
- 감사보고서(별도) 2024·2025: 단위 천원, 둘째 칸이 주석 번호, 한 기간에 금액 칸 두 개(내역·합계),
  감사의견 문단에 재무제표 이름이 나열됨, 주석 표에도 '부채총계'가 있음. 2023 감사보고서는 없음.
  2025 자본총계는 사업보고서 API 값과 일부러 다르게 넣음(대조 불일치 예시)

회사 '00000002'(가상 회사 라마바)는 연결 작성 회사다.
- 2025 연결감사보고서: 단위 백만원, 포괄손익계산서에 '지배기업 소유주지분'이 순이익 귀속·총포괄이익 귀속 두 번 나옴
- 2025 사업보고서 연결 API: account_id가 있음
숫자는 모두 가상이다.
"""
import io
import json
import zipfile

import pandas as pd

CORP = "00000001"
STOCK = "012345"
CORP2 = "00000002"
AUDIT_2024 = "20250310000901"
AUDIT_2025 = "20260309000902"
CONSOL_AUDIT_2025 = "20260311000903"
DECISION_RCEPT = "20260210800123"
VALUEUP_RCEPT = "20250530800456"


def _ok(rows, **extra) -> bytes:
    return json.dumps({"status": "000", "message": "정상", "list": rows, **extra}, ensure_ascii=False).encode()


NO_DATA = json.dumps({"status": "013", "message": "조회된 데이타가 없습니다."}, ensure_ascii=False).encode()
LIMIT = json.dumps({"status": "020", "message": "요청 제한을 초과하였습니다."}, ensure_ascii=False).encode()


def _statement_rows(year: int, net: str, prev: str, liab: str, eq: str, rcept: str) -> list[dict]:
    common = {"rcept_no": rcept, "bsns_year": str(year), "corp_code": CORP, "currency": "KRW"}
    return [
        {**common, "sj_div": "BS", "account_id": "ifrs-full_Liabilities", "account_nm": "부채총계",
         "thstrm_amount": liab, "frmtrm_amount": "", "ord": "1"},
        {**common, "sj_div": "BS", "account_id": "ifrs-full_Equity", "account_nm": "자본총계",
         "thstrm_amount": eq, "frmtrm_amount": "", "ord": "2"},
        {**common, "sj_div": "IS", "account_id": "-표준계정코드 미사용-", "account_nm": "당기순이익(손실)",
         "thstrm_amount": net, "frmtrm_amount": prev, "ord": "3"},
        {**common, "sj_div": "CF", "account_id": "dart_DividendsPaid", "account_nm": "배당금의 지급",
         "thstrm_amount": "(1,500,000,000)", "frmtrm_amount": "", "ord": "4"},
    ]


def _dividend_rows(rcept: str, cash_total: tuple[str, str, str]) -> list[dict]:
    t, f, lw = cash_total
    base = {"rcept_no": rcept, "corp_code": CORP, "corp_name": "가나다"}
    return [
        {**base, "se": "(별도)당기순이익(백만원)", "stock_knd": "", "thstrm": "5,000", "frmtrm": "4,000", "lwfr": "3,000"},
        {**base, "se": "현금배당금총액(백만원)", "stock_knd": "", "thstrm": t, "frmtrm": f, "lwfr": lw},
        {**base, "se": "(별도)현금배당성향(%)", "stock_knd": "", "thstrm": "30.0", "frmtrm": "-", "lwfr": "-"},
        {**base, "se": "주당 현금배당금(원)", "stock_knd": "보통주", "thstrm": "500", "frmtrm": "450", "lwfr": "400"},
        {**base, "se": "주당 현금배당금(원)", "stock_knd": "우선주", "thstrm": "505", "frmtrm": "455", "lwfr": "405"},
    ]


DECISION_XML = """<?xml version="1.0" encoding="utf-8"?>
<DOCUMENT><BODY><TABLE>
<TR><TD>1. 배당구분</TD><TD>결산배당</TD></TR>
<TR><TD>2. 배당종류</TD><TD>현금배당</TD></TR>
<TR><TD ROWSPAN="2">3. 1주당 배당금(원)</TD><TD>보통주식</TD><TE>1,500</TE></TR>
<TR><TD>종류주식</TD><TE>1,505</TE></TR>
<TR><TD ROWSPAN="2">4. 시가배당율(%)</TD><TD>보통주식</TD><TE>3.2</TE></TR>
<TR><TD>종류주식</TD><TE>3.4</TE></TR>
<TR><TD>5. 배당금총액(원)</TD><TE>1,512,345,000</TE></TR>
<TR><TD>6. 배당기준일</TD><TE>2025-12-31</TE></TR>
</TABLE></BODY></DOCUMENT>"""


def _zip(files: dict[str, str]) -> bytes:
    buf = io.BytesIO()
    with zipfile.ZipFile(buf, "w") as z:
        for name, text in files.items():
            z.writestr(name, text)
    return buf.getvalue()


CORPCODE_XML = f"""<?xml version="1.0" encoding="UTF-8"?>
<result>
<list><corp_code>{CORP}</corp_code><corp_name>가나다</corp_name><stock_code>{STOCK}</stock_code><modify_date>20260101</modify_date></list>
<list><corp_code>00000002</corp_code><corp_name>라마바</corp_name><stock_code>067890</stock_code><modify_date>20260101</modify_date></list>
<list><corp_code>00000003</corp_code><corp_name>비상장사</corp_name><stock_code> </stock_code><modify_date>20260101</modify_date></list>
<list><corp_code>00000004</corp_code><corp_name>인지디스플레이</corp_name><stock_code>037330</stock_code><modify_date>20260101</modify_date></list>
<list><corp_code>00000005</corp_code><corp_name>현대자동차</corp_name><stock_code>005380</stock_code><modify_date>20260101</modify_date></list>
<list><corp_code>00000006</corp_code><corp_name>GS</corp_name><stock_code>078930</stock_code><modify_date>20260101</modify_date></list>
<list><corp_code>00000007</corp_code><corp_name>GS리테일</corp_name><stock_code>007070</stock_code><modify_date>20260101</modify_date></list>
<list><corp_code>00000008</corp_code><corp_name>신영증권</corp_name><stock_code>001720</stock_code><modify_date>20260101</modify_date></list>
<list><corp_code>00000009</corp_code><corp_name>인지디스플레이테크</corp_name><stock_code>099990</stock_code><modify_date>20260101</modify_date></list>
</result>"""

# 기업개황(company.json): 법인구분 Y 유가·K 코스닥, 결산월, 업종코드(64~66 금융업)
COMPANY = {
    CORP: ("가나다", "가나다", STOCK, "K", "264", "12"),
    "00000002": ("라마바", "라마바", "067890", "Y", "301", "12"),
    "00000004": ("인지디스플레이", "인지디스플레", "037330", "K", "262", "12"),
    "00000005": ("현대자동차", "현대차", "005380", "Y", "301", "12"),
    "00000006": ("GS", "GS", "078930", "Y", "701", "12"),
    "00000007": ("GS리테일", "GS리테일", "007070", "Y", "471", "12"),
    "00000008": ("신영증권", "신영증권", "001720", "Y", "661", "03"),
    "00000009": ("인지디스플레이테크", "인지디스테크", "099990", "Y", "262", "12"),  # 이름은 비슷하지만 유가
}


_HEAD = ("<TR><TD>과 목</TD><TD>주 석</TD><TD>제 {n}(당) 기</TD><TD></TD>"
         "<TD>제 {p}(전) 기</TD><TD></TD></TR>")


def _audit_xml(n: int, debt: str, equity: str, net: str, prev_net: str, paid: str) -> str:
    head = _HEAD.format(n=n, p=n - 1)
    return f"""<?xml version="1.0" encoding="utf-8"?>
<DOCUMENT><BODY>
<P>독립된 감사인의 감사보고서</P>
<P>우리는 재무상태표, 손익계산서, 자본변동표, 현금흐름표 그리고 주석으로 구성된 재무제표를 감사하였습니다.</P>
<P>재 무 상 태 표</P><P>제 {n} 기 {2014 + n}년 12월 31일 현재</P><P>(단위 : 천원)</P>
<TABLE>{head}
<TR><TD>자산총계</TD><TD></TD><TD></TD><TD>30,000,000</TD><TD></TD><TD>28,000,000</TD></TR>
<TR><TD>부채총계</TD><TD>4,5</TD><TD></TD><TD>{debt}</TD><TD></TD><TD>9,000,000</TD></TR>
<TR><TD>자본총계</TD><TD>17</TD><TD></TD><TD>{equity}</TD><TD></TD><TD>19,000,000</TD></TR>
</TABLE>
<P>첨부된 주석은 본 재무제표의 일부입니다.</P>
<P>손 익 계 산 서</P><P>(단위 : 천원)</P>
<TABLE>{head}
<TR><TD>Ⅰ. 매출액</TD><TD>20</TD><TD></TD><TD>50,000,000</TD><TD></TD><TD>45,000,000</TD></TR>
<TR><TD>Ⅹ. 당기순이익(손실)</TD><TD></TD><TD></TD><TD>{net}</TD><TD></TD><TD>{prev_net}</TD></TR>
</TABLE>
<P>현 금 흐 름 표</P><P>(단위 : 천원)</P>
<TABLE>{head}
<TR><TD>배당금의 지급</TD><TD>25</TD><TD>{paid}</TD><TD></TD><TD>(1,200,000)</TD><TD></TD></TR>
</TABLE>
<P>주석</P><P>1. 일반사항</P>
<TABLE><TR><TD>부채총계</TD><TD>999</TD></TR></TABLE>
</BODY></DOCUMENT>"""


CONSOL_AUDIT_XML = """<?xml version="1.0" encoding="utf-8"?>
<DOCUMENT><BODY>
<P>연 결 재 무 상 태 표</P><P>(단위 : 백만원)</P>
<TABLE><TR><TD>과 목</TD><TD>주 석</TD><TD>당기</TD><TD>전기</TD></TR>
<TR><TD>부채총계</TD><TD>10</TD><TD>8,000</TD><TD>7,500</TD></TR>
<TR><TD>지배기업 소유주지분</TD><TD></TD><TD>11,000</TD><TD>10,000</TD></TR>
<TR><TD>비지배지분</TD><TD></TD><TD>1,000</TD><TD>900</TD></TR>
<TR><TD>자본총계</TD><TD></TD><TD>12,000</TD><TD>10,900</TD></TR>
</TABLE>
<P>연 결 포 괄 손 익 계 산 서</P><P>(단위 : 백만원)</P>
<TABLE><TR><TD>과 목</TD><TD>주 석</TD><TD>당기</TD><TD>전기</TD></TR>
<TR><TD>당기순이익</TD><TD></TD><TD>2,000</TD><TD>1,800</TD></TR>
<TR><TD>지배기업의 소유주에게 귀속되는 당기순이익</TD><TD></TD><TD>1,700</TD><TD>1,500</TD></TR>
<TR><TD>비지배지분에 귀속되는 당기순이익</TD><TD></TD><TD>300</TD><TD>300</TD></TR>
<TR><TD>총포괄이익</TD><TD></TD><TD>2,200</TD><TD>1,900</TD></TR>
<TR><TD>지배기업 소유주지분</TD><TD></TD><TD>1,850</TD><TD>1,580</TD></TR>
</TABLE>
</BODY></DOCUMENT>"""


def _consol_rows() -> list[dict]:
    common = {"rcept_no": "20260316000333", "bsns_year": "2025", "corp_code": CORP2, "currency": "KRW"}
    rows = [
        ("BS", "ifrs-full_Liabilities", "부채총계", "8,000,000,000"),
        ("BS", "ifrs-full_EquityAttributableToOwnersOfParent", "지배기업 소유주지분", "11,000,000,000"),
        ("BS", "ifrs-full_Equity", "자본총계", "12,000,000,000"),
        ("CIS", "ifrs-full_ProfitLoss", "당기순이익", "2,000,000,000"),
        ("CIS", "ifrs-full_ProfitLossAttributableToOwnersOfParent", "지배기업 소유주지분", "1,700,000,000"),
        ("CIS", "ifrs-full_ComprehensiveIncomeAttributableToOwnersOfParent", "지배기업 소유주지분", "1,850,000,000"),
    ]
    return [{**common, "sj_div": sj, "account_id": i, "account_nm": n, "thstrm_amount": v, "frmtrm_amount": ""}
            for sj, i, n, v in rows]


def _audit_list(corp: str, detail: str) -> bytes:
    rows = []
    if corp == CORP and detail == "F001":
        rows = [
            {"corp_code": CORP, "report_nm": "감사보고서 (2024.12)", "rcept_no": AUDIT_2024, "rcept_dt": "20250310"},
            {"corp_code": CORP, "report_nm": "감사보고서 (2025.12)", "rcept_no": AUDIT_2025, "rcept_dt": "20260309"},
        ]
    if corp == CORP2 and detail == "F002":
        rows = [{"corp_code": CORP2, "report_nm": "연결감사보고서 (2025.12)", "rcept_no": CONSOL_AUDIT_2025,
                 "rcept_dt": "20260311"}]
    return _ok(rows, page_no=1, total_page=1, total_count=len(rows)) if rows else NO_DATA


def fake_dart_fetch(url: str, params: dict) -> bytes:
    """OpenDART 주소·파라미터를 보고 가짜 응답을 고른다. 모르는 요청은 013."""
    endpoint = url.rsplit("/", 1)[-1]
    year = str(params.get("bsns_year", ""))
    if endpoint == "corpCode.xml":
        return _zip({"CORPCODE.xml": CORPCODE_XML})
    if endpoint == "company.json":
        if params["corp_code"] not in COMPANY:
            return NO_DATA
        name, short, code, cls, induty, month = COMPANY[params["corp_code"]]
        return json.dumps({"status": "000", "message": "정상", "corp_name": name, "stock_name": short, "stock_code": code,
                           "corp_cls": cls, "induty_code": induty, "acc_mt": month}, ensure_ascii=False).encode()
    if endpoint == "list.json" and params.get("pblntf_ty") == "F":
        return _audit_list(params.get("corp_code"), params.get("pblntf_detail_ty"))
    if endpoint == "document.xml":
        audits = {AUDIT_2024: _audit_xml(10, "10,000,000", "20,000,000", "4,800,000", "3,900,000", "(1,500,000)"),
                  AUDIT_2025: _audit_xml(11, "11,000,000", "21,500,000", "5,100,000", "4,750,000", "(1,500,000)"),
                  CONSOL_AUDIT_2025: CONSOL_AUDIT_XML}
        if params["rcept_no"] in audits:
            return _zip({f"{params['rcept_no']}.xml": audits[params["rcept_no"]]})
    if params.get("corp_code") == CORP2:
        if endpoint == "fnlttSinglAcntAll.json" and params["fs_div"] == "CFS" and year == "2025":
            return _ok(_consol_rows())
        return NO_DATA
    if params.get("corp_code") not in (CORP, None):
        return NO_DATA
    if endpoint == "fnlttSinglAcntAll.json":
        if params["fs_div"] == "CFS":
            return NO_DATA  # 자회사가 없어 연결재무제표가 없다
        if year == "2024":
            return _ok(_statement_rows(2024, "4,800,000,000", "3,900,000,000", "10,000,000,000", "20,000,000,000",
                                       "20250315000111"))
        if year == "2025":
            return _ok(_statement_rows(2025, "5,100,000,000", "4,750,000,000", "11,000,000,000", "21,000,000,000",
                                       "20260316000222"))
        return NO_DATA
    if endpoint == "fnlttSinglAcnt.json":
        return NO_DATA
    if endpoint == "alotMatter.json":
        if year == "2024":
            return _ok(_dividend_rows("20250315000111", ("-", "1,200", "1,000")))
        if year == "2025":
            return _ok(_dividend_rows("20260316000222", ("1,512", "1,400", "-")))
        return NO_DATA
    if endpoint == "stockTotqySttus.json":
        rows = [
            {"rcept_no": "20260316000222", "se": "보통주", "istc_totqy": "1,000,000", "tesstk_co": "8,000", "distb_stock_co": "-"},
            {"rcept_no": "20260316000222", "se": "우선주", "istc_totqy": "10,000", "tesstk_co": "0", "distb_stock_co": "10,000"},
            {"rcept_no": "20260316000222", "se": "합계", "istc_totqy": "1,010,000", "tesstk_co": "8,000", "distb_stock_co": "-"},
        ]
        return _ok(rows)
    if endpoint == "hyslrSttus.json":
        rows = [
            {"rcept_no": "20260316000222", "nm": "홍길동", "relate": "본인", "stock_knd": "보통주", "trmend_posesn_stock_qota_rt": "30.10"},
            {"rcept_no": "20260316000222", "nm": "홍길순", "relate": "친인척", "stock_knd": "보통주", "trmend_posesn_stock_qota_rt": "5.25"},
            {"rcept_no": "20260316000222", "nm": "홍길동", "relate": "본인", "stock_knd": "우선주", "trmend_posesn_stock_qota_rt": "1.00"},
        ]
        return _ok(rows)
    if endpoint == "list.json":
        rows = [
            {"corp_code": CORP, "corp_name": "가나다", "report_nm": "현금ㆍ현물배당결정", "rcept_no": DECISION_RCEPT,
             "rcept_dt": "20260210", "rm": "유"},
            {"corp_code": CORP, "corp_name": "가나다", "report_nm": "[기재정정]기업가치 제고 계획(자율공시)",
             "rcept_no": VALUEUP_RCEPT, "rcept_dt": "20250530", "rm": "유"},
            {"corp_code": CORP, "corp_name": "가나다", "report_nm": "사업보고서 (2025.12)", "rcept_no": "20260316000222",
             "rcept_dt": "20260316", "rm": ""},
        ]
        page = int(params.get("page_no", 1))
        return _ok(rows[:2] if page == 1 else rows[2:], page_no=page, total_page=2, total_count=3)
    if endpoint == "document.xml":
        if params["rcept_no"] == DECISION_RCEPT:
            return _zip({f"{DECISION_RCEPT}.xml": DECISION_XML})
        return b'<?xml version="1.0" encoding="UTF-8"?><result><status>014</status><message>file not found</message></result>'
    return NO_DATA


class FakePrices:
    """pykrx 대신 쓰는 가짜 시세. 012345는 1월 6~7일 거래정지, 067890은 1월 8일 상장."""

    name = "가짜 시세"

    def __init__(self):
        self.days = pd.to_datetime(["2026-01-02", "2026-01-05", "2026-01-06", "2026-01-07", "2026-01-08", "2026-01-09"])

    def index(self, code, start, end):
        return pd.DataFrame({"시가": 2500.0, "고가": 2510.0, "저가": 2490.0,
                             "종가": [2500.0, 2510.0, 2490.0, 2520.0, 2530.0, 2540.0], "거래량": 1_000_000}, index=self.days)

    def tickers(self, date, market):
        return {"012345": "가나다"} if market == "KOSDAQ" else {"067890": "라마바"}

    def prices(self, ticker, start, end):
        if ticker == "012345":
            return pd.DataFrame({
                "시가": [10000, 10100, 0, 0, 10300, 10400], "고가": [10100, 10200, 0, 0, 10400, 10500],
                "저가": [9900, 10000, 0, 0, 10200, 10300], "종가": [10000, 10100, 10100, 10100, 10350, 10450],
                "거래량": [500, 600, 0, 0, 700, 800]}, index=self.days)
        if ticker == "067890":
            d = self.days[4:]
            return pd.DataFrame({"시가": [5000, 5100], "고가": [5200, 5200], "저가": [4900, 5000],
                                 "종가": [5100, 5150], "거래량": [9000, 8000]}, index=d)
        return pd.DataFrame()


# 국가법령정보센터 응답: 검색 결과가 하나라 law가 dict, 항이 하나라 dict, 호는 목록
LAW_SEARCH = {"LawSearch": {"law": {"법령일련번호": "999999", "법령명한글": "조세특례제한법", "시행일자": "20260918",
                                    "현행연혁코드": "현행"}}}
LAW_BODY = {"법령": {"기본정보": {"법령명_한글": "조세특례제한법"}, "조문": {"조문단위": [
    {"조문번호": "104", "조문가지번호": "26", "조문여부": "조문", "조문제목": "다른 조문", "조문내용": "제104조의26(다른 조문)"},
    {"조문번호": "104", "조문여부": "전문", "조문내용": "제5절 ..."},
    {"조문번호": "104", "조문가지번호": "27", "조문여부": "조문", "조문제목": "고배당기업 주식 배당소득에 대한 과세특례",
     "조문내용": "제104조의27(고배당기업 주식 배당소득에 대한 과세특례)",
     "항": {"항번호": "①", "항내용": [["① 다음 각 호의 요건을 모두 갖춘 내국법인 ..."]],
            "호": [{"호번호": "1.", "호내용": "1. 가상 요건 하나"}, {"호번호": "2.", "호내용": "2. 가상 요건 둘",
                                                                "목": {"목번호": "가.", "목내용": "가. 가상 세부 요건"}}]}},
]}}}


def fake_law_fetch(url: str, params: dict) -> bytes:
    if params.get("OC") == "wrong":
        return "<html>인증 오류</html>".encode()
    if url.endswith("lawSearch.do"):
        return json.dumps(LAW_SEARCH, ensure_ascii=False).encode()
    return json.dumps(LAW_BODY, ensure_ascii=False).encode()


# KIND '밸류업 고배당기업' 내려받기와 같은 모양(HTML 표, 종목코드 열 없음, 회사명은 KRX 약칭)
KIND_HTML = """<html><body><table>
<tr><th>번호</th><th>시장구분</th><th>회사명</th><th>공시제목</th><th>사업연도</th><th>결산월</th><th>배당성향(%)</th><th>이익배당금증가율(%)</th></tr>
<tr><td>1</td><td>코스닥</td><td>인지디스플레</td><td>기업가치 제고 계획(자율공시)</td><td>2025</td><td>12</td><td>27.76</td><td>52.36</td></tr>
<tr><td>2</td><td>유가</td><td>현대차</td><td>기업가치 제고 계획(자율공시)</td><td>2025</td><td>12</td><td>25.1</td><td>10.2</td></tr>
<tr><td>3</td><td>유가</td><td>GS</td><td>기업가치 제고 계획(자율공시)</td><td>2025</td><td>12</td><td>40.2</td><td>0</td></tr>
<tr><td>4</td><td>유가</td><td>신영증권</td><td>기업가치 제고 계획(자율공시)</td><td>2026</td><td>03</td><td>40.3</td><td>5</td></tr>
<tr><td>5</td><td>코스닥</td><td>가나다</td><td>[정정] 기업가치 제고 계획(자율공시)</td><td>2025</td><td>12</td><td>30.0</td><td>12</td></tr>
<tr><td>6</td><td>유가</td><td>라마바</td><td>기업가치 제고 계획(자율공시)</td><td>2025</td><td>12</td><td>45.0</td><td>10</td></tr>
<tr><td>7</td><td>코스닥</td><td>없는회사</td><td>기업가치 제고 계획(자율공시)</td><td>2025</td><td>12</td><td>-</td><td>-</td></tr>
</table></body></html>"""

In [ ]:
# @title 1-2. 불러오기
import os, sys
sys.path.insert(0, os.getcwd())
import pandas as pd
pd.set_option("display.max_colwidth", 80)
from core.sources import samples
from core.sources.base import Client, run_resumable, status_table, secret, forget_secret
print("코드 준비 완료")

## 2. 연결·키 확인 (실패하면 여기서 멈춤)

- 구글 드라이브를 연결하고 내 드라이브 `EXPORT_FOLDER`(없으면 만듦)에 시험 파일을 써 봅니다.
- 진행 상황(이어받기 기록)과 응답 원본은 **내 드라이브** `ONE_PICK_data/{NB_NAME}/`에 둡니다(다음 날 이어서 돌리기 위해).
- 키는 코랩 🔑 **보안 비밀**에 같은 이름으로 넣어 두면 묻지 않습니다. 없으면 화면에 보이지 않는 입력 칸이 뜹니다.
  키는 메모리에만 있고 파일·출력·zip에 남지 않습니다. 잘못 넣었으면 `forget_secret("이름")` 실행 후 이 셀을 다시 실행하세요.

- `KRX_ID`(KRX 정보데이터시스템 아이디): https://data.krx.co.kr 회원가입
- `KRX_PW`(KRX 정보데이터시스템 비밀번호): 위 계정의 비밀번호

In [ ]:
# @title 2-1. 드라이브 연결 · 저장 폴더 확인
import datetime as _dt
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:  # 코랩이 아닌 곳(시험용)
    IN_COLAB = False

DRIVE = Path("/content/drive/MyDrive") if IN_COLAB else Path("my_drive")
EXPORT = DRIVE / EXPORT_FOLDER
EXPORT.mkdir(parents=True, exist_ok=True)
OUT = DRIVE / "ONE_PICK_data" / NB_NAME

probe = EXPORT / f".쓰기시험_{NB_NAME}"
try:
    probe.write_text("ok"); probe.unlink()
except OSError as error:
    raise RuntimeError(f"저장 폴더에 쓸 수 없습니다(드라이브 용량·연결 확인): {EXPORT} · {error}")
OUT.mkdir(parents=True, exist_ok=True)
RAW = OUT / "raw"
RESULT = OUT / "결과"
RESULT.mkdir(exist_ok=True)
print("결과 zip 폴더:", EXPORT)
print("내 작업 폴더:", OUT)

In [ ]:
# @title 2-2. 키 받기 · 실제로 한 번 불러 확인
secret("KRX_ID", "KRX 정보데이터시스템 아이디 입력(화면에 보이지 않음): ")
secret("KRX_PW", "KRX 정보데이터시스템 비밀번호 입력(화면에 보이지 않음): ")
os.environ["KRX_ID"], os.environ["KRX_PW"] = secret("KRX_ID"), secret("KRX_PW")  # pykrx를 처음 부르기 전에 넣어야 로그인됨
from core.sources.krx import MARKET_NAMES, listing, market_panels
check = listing(END)
if not check.found:
    raise RuntimeError("KRX 종목 목록을 받지 못했습니다(로그인·휴장일 확인): " + " → ".join(check.attempts))
print("KRX 확인 완료:", " · ".join(check.attempts))

## 3. 대상 회사 (company_map.csv)

`01_기업_매핑표_만들기` 담당자에게 받은 `company_map.csv`를 씁니다. 다음 순서로 찾고, 없으면 업로드 창을 띄웁니다.
코랩 왼쪽 📁에 직접 올린 파일 → 내 드라이브 `EXPORT_FOLDER` → 내 작업 폴더. `매핑 상태`가 '정상'인 회사만 수집합니다.

In [ ]:
# @title 3-1. 매핑표 읽기
places = [Path(MAP_FILE), Path("/content") / MAP_FILE, EXPORT / MAP_FILE, OUT / MAP_FILE]
map_path = next((p for p in places if p.is_file()), None)
if map_path is None:
    if not IN_COLAB:
        raise FileNotFoundError(f"{MAP_FILE}이 없습니다: {[str(p) for p in places]}")
    from google.colab import files
    print(f"{MAP_FILE}을 찾지 못했습니다. 업로드해 주세요.")
    up = files.upload()
    map_path = OUT / next(iter(up))
    map_path.write_bytes(next(iter(up.values())))
company_map = pd.read_csv(map_path, dtype=str)
targets = company_map[company_map["매핑 상태"] == "정상"].reset_index(drop=True)
skipped = company_map[company_map["매핑 상태"] != "정상"]
print("매핑표:", map_path, "·", len(company_map), "곳 중 대상", len(targets), "곳")
if len(skipped):
    print(f"고유번호가 없어 빠지는 회사 {len(skipped)}곳:", ", ".join(skipped["회사명"].head(20)))
by_market = {}
for _, r in targets.iterrows():
    by_market.setdefault(MARKET_NAMES.get(r["시장구분"], "KOSPI"), []).append(r["종목코드"])
key_map = dict(zip(targets["종목코드"], targets["corp_code"]))  # 열 이름을 corp_code로 (이벤트 스터디와 맞춤)
print({m: len(v) for m, v in by_market.items()})

## 4. 수집

종목이 많으면 오래 걸립니다(639종목 × 8년이면 수십 분).

In [ ]:
# @title 4-1. 수집
from core.sources.krx import NORMAL, classify_days
panel = market_panels(by_market, START, END, key_map=key_map)
panel["prices"].to_csv(RESULT / "prices.csv", encoding="utf-8-sig")
for m, series in panel["markets"].items():
    series.to_frame(f"{m} 종가").to_csv(RESULT / f"market_{m}.csv", encoding="utf-8-sig")
pd.Series(panel["market_of"], name="시장").rename_axis("corp_code").to_csv(RESULT / "종목별_시장.csv", encoding="utf-8-sig")
panel["report"].to_csv(RESULT / "report.csv", index=False, encoding="utf-8-sig")

rows = []
for code, f in panel["fetched"].items():
    days = classify_days(f.value, panel["prices"].index)
    bad = days[days["상태"] != NORMAL]
    rows.append(bad.assign(종목코드=code).reset_index(names="거래일")[["종목코드", "거래일", "종가", "상태"]])
status_days = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=["상태"])
status_days.to_csv(RESULT / "일별상태.csv", index=False, encoding="utf-8-sig")
display(panel["report"].head(20))
display(status_days["상태"].value_counts().rename("일수").to_frame())

## 5. 내보내기 (zip → 내 드라이브)

`결과/` 폴더(prices·market·종목별_시장·report·일별상태)를 zip 하나로 묶어 내 드라이브 `EXPORT_FOLDER`에 `<노트북 이름>_<날짜_시간>.zip`으로 넣습니다. 응답 원본(`raw/`)과 진행 기록은 넣지 않습니다.
팀에 넘길 때는 이 zip을 직접 공유 폴더나 메신저에 올려 주세요.

In [ ]:
# @title 5-1. zip 내보내기
import shutil, tempfile
stamp = _dt.datetime.now().strftime("%Y%m%d_%H%M")
with tempfile.TemporaryDirectory() as tmp:
    made = shutil.make_archive(str(Path(tmp) / f"{NB_NAME}_{stamp}"), "zip", root_dir=RESULT)
    target = EXPORT / Path(made).name
    shutil.copyfile(made, target)
names = sorted(p.relative_to(RESULT).as_posix() for p in RESULT.rglob("*") if p.is_file())
print("내보냄:", target)
print("들어간 파일", len(names), "개:", names[:15], "..." if len(names) > 15 else "")